In [0]:
%sql
-- ESTO SE DEBE EJECUTAR EN EL PIPELINE, ESTA EN EL NOTEBOOK
-- 
CREATE OR REFRESH STREAMING TABLE ORDER_BRONZE(
    CONSTRAINT ID_NO_NULO EXPECT (id IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT AGE_RANGO_LOGICO EXPECT (age >= 0 AND age < 120) ON VIOLATION DROP ROW
)
COMMENT "TABLA DE ATERRIZAJE"
TBLPROPERTIES (
  "delta.enableChangeDataFeed" = "true",
  "pipelines.metastore.tableName" = "table_destino",
  "my_custom_metadata.layer" = "bronze",
  -- 1. Habilita que se puedan hacer limpiezas (VACUUM) de archivos muy recientes (por debajo de 168 horas / 7 días)
  "delta.compatibility.symlinkFormatManifest.enabled" = "false",
  
  -- 2. Define el tiempo de retención de logs para el borrado (ej. 7 días)
  "delta.logRetentionDuration" = "interval 7 days",
  
  -- 3. Define la retención de los archivos de datos físicamente eliminados
  "delta.deletedFileRetentionDuration" = "interval 7 days"
)
AS SELECT * FROM STREAM read_files(
    '/Volumes/main/default/my-volume/orders',
    format => "json",
    schemaHints => "id INT, name STRING, age INT",
    maxFilesPerTrigger => 10,
    maxBytesPerTrigger => "10m",
    useNotifications => false,
    includeExistingFiles => true
);

In [0]:
%sql
-- Esto se ejecuta como una tarea de mantenimiento separada, NO en el stream
OPTIMIZE main.default.table_destino 
ZORDER BY (id);

VACUUM main.default.table_destino RETAIN 168 HOURS; -- Retiene 7 días de historial